In [1]:
import os
import csv
import math
import logging
from pathlib import Path
from typing import List, Optional

import numpy as np
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.anchor_utils import AnchorGenerator
from torchmetrics.detection import MeanAveragePrecision


In [2]:
DINOV3_GITHUB_LOCATION = "/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/Foundation-Models/dinov3"
DINOV3_LOCATION = os.getenv("DINOV3_LOCATION") or DINOV3_GITHUB_LOCATION
DINO_MODEL_NAME = "dinov3_vitl16"
DINO_WEIGHTS = "/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/Foundation-Models/dinov3/dinov3_vitl16_pretrain_sat493m-eadcf0ff.pth"


In [3]:
import math
import torch
import torch.nn as nn

# ----- your wrapper (with safe prints for ints) -----
class DinoV3BackboneWrapper(nn.Module):
    """Return {'0': Tensor[B, C, H/16, W/16]} with out_channels=C."""
    def __init__(self, dino_model: nn.Module, patch_stride: int = 16):
        super().__init__()
        self.dino = dino_model
        self.patch_stride = patch_stride
        C = getattr(dino_model, "embed_dim", None) or getattr(dino_model, "num_features", None)
        if C is None:
            with torch.no_grad():
                x = torch.zeros(1, 3, 32, 32)
                tokens, Ht, Wt = self._get_patch_tokens(x)
                C = tokens.shape[-1]
        self.out_channels = C

    @torch.no_grad()
    def _maybe_h_w(self, x):
        _, _, H, W = x.shape
        return math.ceil(H / self.patch_stride), math.ceil(W / self.patch_stride)

    def _get_patch_tokens(self, x):
        try:
            out = self.dino.forward_features(x)
            print("forward_features type:", type(out))

            if isinstance(out, dict):
                if "x_norm_patchtokens" in out:
                    tokens = out["x_norm_patchtokens"]
                    Ht = out.get("H") or self._maybe_h_w(x)[0]
                    Wt = out.get("W") or self._maybe_h_w(x)[1]
                    print("tokens from dict['x_norm_patchtokens']:", tokens.shape)
                    print(f"Ht, Wt: {Ht}, {Wt}")
                    return tokens, Ht, Wt
                if "tokens" in out and out["tokens"] is not None:
                    t = out["tokens"]
                    Ht, Wt = self._maybe_h_w(x)
                    if t.shape[1] == (Ht * Wt + 1):
                        t = t[:, 1:, :]
                    return t, Ht, Wt

            if isinstance(out, torch.Tensor):
                t = out
                Ht, Wt = self._maybe_h_w(x)
                N = Ht * Wt
                if t.shape[1] == N + 1:
                    t = t[:, 1:, :]
                elif t.shape[1] != N:
                    N = t.shape[1]
                    Wt = int(round(math.sqrt(N)))
                    Ht = N // Wt
                return t, Ht, Wt
        except Exception:
            pass

        if hasattr(self.dino, "get_intermediate_layers"):
            t = self.dino.get_intermediate_layers(x, n=1, return_class_token=False)[0]
            Ht, Wt = self._maybe_h_w(x)
            return t, Ht, Wt

        t = self.dino(x)
        Ht, Wt = self._maybe_h_w(x)
        if t.dim() == 3 and t.shape[1] == (Ht * Wt + 1):
            t = t[:, 1:, :]
        return t, Ht, Wt

    def forward(self, x: torch.Tensor):
        tokens, Ht, Wt = self._get_patch_tokens(x)
        print("Tokens shape:", tokens.shape)
        print(f"Ht, Wt: {Ht}, {Wt}")
        B, N, C = tokens.shape
        feat = tokens.transpose(1, 2).contiguous().view(B, C, Ht, Wt)
        print("Feature map shape:", feat.shape)
        return {"0": feat}



In [4]:
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.anchor_utils import AnchorGenerator
from torchvision.ops import MultiScaleRoIAlign

def create_model(dino_model, num_classes: int, image_size: int = 800):
    backbone = DinoV3BackboneWrapper(dino_model, patch_stride=16)
    anchor_generator = AnchorGenerator(
        sizes=((16,32, 64, 128, 256),), 
        aspect_ratios=((0.5, 1.0, 2.0),)
    )

    model = FasterRCNN(
        backbone=backbone,
        num_classes=num_classes,
        rpn_anchor_generator=anchor_generator,
        min_size=image_size,
        max_size=image_size,
    )
    return model

In [5]:
dino_model = torch.hub.load(
        repo_or_dir=DINOV3_LOCATION,
        model=DINO_MODEL_NAME,
        source="local",
        weights=DINO_WEIGHTS,
        skip_validation=True,
    )

In [15]:
device=torch.device('cuda:2' if torch.cuda.is_available() else 'cpu')
model=create_model(dino_model, num_classes=4, image_size=800)
model

FasterRCNN(
  (transform): GeneralizedRCNNTransform(
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
      Resize(min_size=(800,), max_size=800, mode='bilinear')
  )
  (backbone): DinoV3BackboneWrapper(
    (dino): DinoVisionTransformer(
      (patch_embed): PatchEmbed(
        (proj): Conv2d(3, 1024, kernel_size=(16, 16), stride=(16, 16))
        (norm): Identity()
      )
      (rope_embed): RopePositionEmbedding()
      (blocks): ModuleList(
        (0-23): 24 x SelfAttentionBlock(
          (norm1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (attn): SelfAttention(
            (qkv): LinearKMaskedBias(in_features=1024, out_features=3072, bias=True)
            (attn_drop): Dropout(p=0.0, inplace=False)
            (proj): Linear(in_features=1024, out_features=1024, bias=True)
            (proj_drop): Dropout(p=0.0, inplace=False)
          )
          (ls1): LayerScale()
          (norm2): LayerNorm((1024,), eps=1e-05, elementwise_affine=

In [ ]:

# -------------------------
# Dataset (PNG RGB only) + YOLO-OBB (9-tuple) -> AABB
# -------------------------
class BrickKilnDataset(Dataset):
    def __init__(self, root: str, split: str, input_size: int = 800):
        self.root = Path(root)
        self.split = split
        self.img_dir = self.root / "images"
        self.label_dir = self.root / "yolo_obb_labels"

        # Keep as [0,1], no ImageNet normalization (works better with learned 1x1 RGB->12 adapter)
        self.transform = transforms.Compose([
            transforms.Resize((input_size, input_size)),
            transforms.ToTensor(),
        ])

        self.img_files = []
        all_files = sorted([f for f in os.listdir(self.img_dir) if f.lower().endswith(".png")])
        logging.info(f"Scanning {len(all_files)} PNGs in {self.img_dir}...")
        for img_name in tqdm(all_files, desc=f"Verify {split} data"):
            if self._has_valid_annotations(img_name):
                self.img_files.append(img_name)
        logging.info(f"Found {len(self.img_files)} valid images in {self.img_dir}")

    def _has_valid_annotations(self, img_name: str) -> bool:
        label_path = self.label_dir / f"{Path(img_name).stem}.txt"
        if not label_path.exists():
            return False
        with open(label_path, 'r') as f:
            for line in f:
                if len(line.strip().split()) == 9:
                    return True
        return False

    def __len__(self):
        return len(self.img_files)

    def __getitem__(self, idx: int):
        img_name = self.img_files[idx]
        img_path = self.img_dir / img_name
        label_path = self.label_dir / f"{Path(img_name).stem}.txt"

        img = Image.open(img_path).convert("RGB")
        img_tensor = self.transform(img)
        _, h, w = img_tensor.shape

        boxes, labels = [], []
        with open(label_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) != 9:
                    continue
                cls_id = int(parts[0]) + 1  # reserve 0 for background
                obb = np.array([float(p) for p in parts[1:]])
                xs, ys = obb[0::2] * w, obb[1::2] * h
                xmin, ymin, xmax, ymax = np.min(xs), np.min(ys), np.max(xs), np.max(ys)
                if xmax > xmin and ymax > ymin:
                    boxes.append([xmin, ymin, xmax, ymax])
                    labels.append(cls_id)

        target = {
            "boxes": torch.as_tensor(boxes, dtype=torch.float32),
            "labels": torch.as_tensor(labels, dtype=torch.int64),
        }
        return img_tensor, target


def collate_fn(batch):
    batch = [item for item in batch if item[1]["boxes"].shape[0] > 0]
    if not batch:
        return None, None
    return tuple(zip(*batch))



In [ ]:

# --------------------------------------
# Final evaluation (IN-REGION / OOR)
# --------------------------------------
@torch.no_grad()
def evaluate_region(model, root: str, split: str, device,
                    batch_size=16, num_workers=8, image_size=800,
                    title="", results_csv=None):
    """
    Computes:
      - CA mAP@50 (class-agnostic; collapse labels to 1)
      - MC mAP@50 (macro over classes WITH GT only)
      - Per-class AP@50 only for classes WITH GT (no -1.0 surprises).
    """
    ds = BrickKilnDataset(root=root, split=split, input_size=image_size)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False,
                    num_workers=num_workers, pin_memory=True,
                    collate_fn=collate_fn)

    model.eval()
    metric_class = MeanAveragePrecision(
        box_format='xyxy', iou_type='bbox', class_metrics=True,  iou_thresholds=[0.5]
    )
    metric_agn   = MeanAveragePrecision(
        box_format='xyxy', iou_type='bbox', class_metrics=False, iou_thresholds=[0.5]
    )

    for batch in tqdm(dl, desc=f"Test [{title or split}]"):
        if batch is None:
            continue
        images, targets = batch
        images = [i.to(device) for i in images]
        preds  = model(images)

        preds_cpu = [{k: v.to('cpu') for k, v in p.items()} for p in preds]
        tgts_cpu  = [{k: v.to('cpu') for k, v in t.items()} for t in targets]

        # class-wise
        metric_class.update(preds_cpu, tgts_cpu)

        # class-agnostic (collapse labels to 1)
        preds_agn = [{'boxes': p['boxes'], 'scores': p['scores'],
                      'labels': torch.ones_like(p['labels'])} for p in preds_cpu]
        tgts_agn  = [{'boxes': t['boxes'],
                      'labels': torch.ones_like(t['labels'])} for t in tgts_cpu]
        metric_agn.update(preds_agn, tgts_agn)

    # ---- Compute ----
    res_class = metric_class.compute()
    res_agn   = metric_agn.compute()

    # CA mAP@50 (explicit)
    ca_map50 = float(res_agn.get('map_50', res_agn.get('map', torch.tensor(0.0)))) * 100.0

    # Per-class AP@50 list and classes
    classes = res_class.get('classes', torch.tensor([])).tolist() if 'classes' in res_class else []
    ap_list = res_class.get('map_per_class', torch.tensor([])).tolist() if 'map_per_class' in res_class else []

    # Filter out undefined classes (torchmetrics uses -1.0 or NaN when no GT)
    valid_pairs = []
    for c, ap in zip(classes, ap_list):
        try:
            apf = float(ap)
        except Exception:
            continue
        if apf >= 0.0 and np.isfinite(apf):
            valid_pairs.append((int(c), apf))

    per_cls = {c: ap * 100.0 for c, ap in valid_pairs}
    if len(valid_pairs) > 0:
        mc_map50 = sum(ap for _, ap in valid_pairs) / len(valid_pairs) * 100.0
    else:
        mc_map50 = 0.0

    # Pretty print (class ids 1,2,3 mapped to CFCBK/FCBK/Zigzag)
    def g(k):  # safe getter in %
        return float(per_cls.get(k, 0.0))

    print("\n" + "=" * 84)
    print(f" Region: {title or (Path(root).name + ' — ' + split)}")
    print("=" * 84)
    print(f"{'CA mAP@50':<12}{'MC mAP@50':<12}{'CFCBK@50':<12}{'FCBK@50':<12}{'Zigzag@50':<12}")
    print("-" * 84)
    print(f"{ca_map50:<12.2f}{mc_map50:<12.2f}{g(1):<12.2f}{g(2):<12.2f}{g(3):<12.2f}")
    print("=" * 84 + "\n")

    # Optional: write one line per region to CSV
    if results_csv is not None:
        is_new = not os.path.exists(results_csv)
        with open(results_csv, "a", newline="") as f:
            w = csv.writer(f)
            if is_new:
                w.writerow(["Region", "Split", "CA_mAP50", "MC_mAP50",
                            "CFCBK_mAP50", "FCBK_mAP50", "Zigzag_mAP50"])
            w.writerow([title or Path(root).name, split,
                        f"{ca_map50:.2f}", f"{mc_map50:.2f}",
                        f"{g(1):.2f}", f"{g(2):.2f}", f"{g(3):.2f}"])

    return ca_map50, mc_map50, per_cls

In [6]:
# Path to your fine-tuned contrastive checkpoint
finetune_ckpt = "/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/Foundation-Models/cvpr/dino_vitl16_metadata_contrastive_lr_epoch46.pth"

if os.path.exists(finetune_ckpt):
    state = torch.load(finetune_ckpt, map_location="cpu")
    # if it has nested dict from Lightning / EMA wrappers
    if isinstance(state, dict) and "state_dict" in state and isinstance(state["state_dict"], dict):
        state = state["state_dict"]

    # Clean key prefixes to align with DINO backbone
    clean = {}
    for k, v in state.items():
        nk = k
        if nk.startswith("module."):
            nk = nk[len("module."):]
        if nk.startswith("backbone.") or nk.startswith("encoder."):
            nk = nk.split(".", 1)[1]
        clean[nk] = v

    # Load non-strict to ignore projector or metadata heads
    res = dino_model.load_state_dict(clean, strict=False)
    print(f"[CKPT] Loaded fine-tuned weights from {finetune_ckpt}")
    print(f"Missing keys: {len(getattr(res, 'missing_keys', []))}, Unexpected: {len(getattr(res, 'unexpected_keys', []))}")
else:
    print(f"[WARN] Fine-tuned checkpoint not found: {finetune_ckpt}")

[CKPT] Loaded fine-tuned weights from /home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/Foundation-Models/cvpr/dino_vitl16_metadata_contrastive_lr_epoch46.pth
Missing keys: 2, Unexpected: 0


In [7]:
import torch

ckpt_path = "/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/Foundation-Models/cvpr/dino_vitl16_metadata_contrastive_lr_epoch46.pth"
state = torch.load(ckpt_path, map_location="cpu")

# Handle nested dicts (Lightning or DDP)
if isinstance(state, dict) and "state_dict" in state and isinstance(state["state_dict"], dict):
    state = state["state_dict"]

print(f"Total keys: {len(state)}")
for k in sorted(state.keys()):
    print(k)

Total keys: 368
blocks.0.attn.proj.bias
blocks.0.attn.proj.weight
blocks.0.attn.qkv.bias
blocks.0.attn.qkv.bias_mask
blocks.0.attn.qkv.weight
blocks.0.ls1.gamma
blocks.0.ls2.gamma
blocks.0.mlp.fc1.bias
blocks.0.mlp.fc1.weight
blocks.0.mlp.fc2.bias
blocks.0.mlp.fc2.weight
blocks.0.norm1.bias
blocks.0.norm1.weight
blocks.0.norm2.bias
blocks.0.norm2.weight
blocks.1.attn.proj.bias
blocks.1.attn.proj.weight
blocks.1.attn.qkv.bias
blocks.1.attn.qkv.bias_mask
blocks.1.attn.qkv.weight
blocks.1.ls1.gamma
blocks.1.ls2.gamma
blocks.1.mlp.fc1.bias
blocks.1.mlp.fc1.weight
blocks.1.mlp.fc2.bias
blocks.1.mlp.fc2.weight
blocks.1.norm1.bias
blocks.1.norm1.weight
blocks.1.norm2.bias
blocks.1.norm2.weight
blocks.10.attn.proj.bias
blocks.10.attn.proj.weight
blocks.10.attn.qkv.bias
blocks.10.attn.qkv.bias_mask
blocks.10.attn.qkv.weight
blocks.10.ls1.gamma
blocks.10.ls2.gamma
blocks.10.mlp.fc1.bias
blocks.10.mlp.fc1.weight
blocks.10.mlp.fc2.bias
blocks.10.mlp.fc2.weight
blocks.10.norm1.bias
blocks.10.norm1.

In [9]:
if os.path.exists(finetune_ckpt):
    state = torch.load(finetune_ckpt, map_location="cpu")
    if isinstance(state, dict) and "state_dict" in state and isinstance(state["state_dict"], dict):
        state = state["state_dict"]

    clean = {}
    for k, v in state.items():
        nk = k
        if nk.startswith("module."):
            nk = nk[len("module."):]
        if nk.startswith("backbone.") or nk.startswith("encoder."):
            nk = nk.split(".", 1)[1]
        clean[nk] = v

    res = dino_model.load_state_dict(clean, strict=False)
    missing = getattr(res, "missing_keys", [])
    unexpected = getattr(res, "unexpected_keys", [])
    print(f"[CKPT] Loaded fine-tuned weights from {finetune_ckpt}")
    print(f"Missing keys ({len(missing)}):")
    for k in missing:
        print("   ", k)
    print(f"Unexpected keys ({len(unexpected)}):")
    for k in unexpected:
        print("   ", k)
else:
    print(f"[WARN] Fine-tuned checkpoint not found: {finetune_ckpt}")

[CKPT] Loaded fine-tuned weights from /home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/Foundation-Models/cvpr/dino_vitl16_metadata_contrastive_lr_epoch46.pth
Missing keys (2):
    local_cls_norm.weight
    local_cls_norm.bias
Unexpected keys (0):
